In [1]:
!pip install xgboost

import pandas as pd
import numpy as np
import os
import torch
import torch.nn as nn
import torch.optim as optim
from google.colab import drive
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, classification_report
import time
import json
from datetime import datetime
import joblib
import xgboost as xgb

# 挂载 Google Drive
print("正在挂载 Google Drive...")
drive.mount('/content/drive')
print("✓ Google Drive 挂载成功！")

# 检查 GPU 是否可用
if torch.cuda.is_available():
    print(f"✓ GPU 已找到！设备名称: {torch.cuda.get_device_name(0)}")
    device = torch.device("cuda")
else:
    print("⚠️ 警告：未找到 GPU，将使用 CPU 进行训练。")
    device = torch.device("cpu")

正在挂载 Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✓ Google Drive 挂载成功！
✓ GPU 已找到！设备名称: NVIDIA A100-SXM4-40GB


In [5]:
import os

# 試著列出 BTAT 資料夾裡的內容
!ls -l /content/drive/MyDrive/BTAT/

total 12
drwx------ 2 root root 4096 Jul  2 04:30 docs
drwx------ 2 root root 4096 Jul  2 07:06 gpu_models_output
drwx------ 2 root root 4096 Jul  2 04:29 Merged_dataset


In [6]:
data_folder_path = "/content/drive/MyDrive/BTAT/Merged_dataset/"
# --- 模型和结果要储存到的地方 ---
output_folder_path = "/content/drive/MyDrive/BTAT/gpu_models_output/"

# 建立输出资料夹
os.makedirs(output_folder_path, exist_ok=True)
print(f"\n资料将从这里读取: {data_folder_path}")
print(f"模型和结果将储存到: {output_folder_path}")

# 读取并合并所有 CSV 档案
files_to_load = ['w1.csv', 'w2.csv', 'w3.csv']
full_file_paths = [os.path.join(data_folder_path, f) for f in files_to_load]

print("\n正在读取并合并资料...")
try:
    df_list = [pd.read_csv(file, header=None) for file in full_file_paths]
    df = pd.concat(df_list, ignore_index=True)
    print(f"✓ 资料合并完成，总共 {len(df)} 笔资料。")
except FileNotFoundError:
    print(f"❌ 错误：在 '{data_folder_path}' 中找不到档案。请检查您的路径是否正确！")
    exit()

# 加上正确的栏位名称
column_names = [
    "duration", "protocol_type", "service", "src_bytes", "dst_bytes", "flag",
    "count", "srv_count", "serror_rate", "same_srv_rate", "diff_srv_rate",
    "srv_serror_rate", "srv_diff_host_rate", "dst_host_count",
    "dst_host_srv_count", "dst_host_same_srv_rate", "dst_host_diff_srv_rate",
    "dst_host_same_src_port_rate", "dst_host_serror_rate",
    "dst_host_srv_diff_host_rate", "dst_host_srv_serror_rate", "label"
]
df.columns = column_names
print("✓ 已成功为资料集加上栏位名称。")


资料将从这里读取: /content/drive/MyDrive/BTAT/Merged_dataset/
模型和结果将储存到: /content/drive/MyDrive/BTAT/gpu_models_output/

正在读取并合并资料...
✓ 资料合并完成，总共 210000 笔资料。
✓ 已成功为资料集加上栏位名称。


In [9]:
# ====================== 新增的診斷程式碼 ======================
print("\n--- 正在檢查標籤分佈情況 ---")
print("在切分資料前，各類標籤的實際數量為：")
print(df['label'].value_counts())
print("--------------------------------------------------\n")
# ==========================================================


--- 正在檢查標籤分佈情況 ---
在切分資料前，各類標籤的實際數量為：
label
Normal    150000
MitM       15000
BP         15000
DoS        15000
FoT        15000
Name: count, dtype: int64
--------------------------------------------------



In [7]:
print("\n🔧 开始进行资料预处理...")

# 分离特征和标签
X = df.drop('label', axis=1)
y = df['label']

# 识别分类和数值特征
categorical_features = X.select_dtypes(include=['object']).columns
numerical_features = X.select_dtypes(include=np.number).columns
print(f"分类特征: {list(categorical_features)}")

# 对分类特征进行 One-Hot 编码
X = pd.get_dummies(X, columns=categorical_features, drop_first=True)
print("✓ 分类特征已完成 One-Hot 编码。")

# 编码标签
le = LabelEncoder()
y_encoded = le.fit_transform(y)
print(f"✓ 标签编码完成。类别: {list(le.classes_)}")

# 标准化数值特征
scaler = StandardScaler()
X[numerical_features] = scaler.fit_transform(X[numerical_features])
print("✓ 数值特征已完成标准化。")
print(f"预处理后，特征资料的形状: {X.shape}")


🔧 开始进行资料预处理...
分类特征: ['protocol_type', 'service', 'flag']
✓ 分类特征已完成 One-Hot 编码。
✓ 标签编码完成。类别: ['BP', 'DoS', 'FoT', 'MitM', 'Normal']
✓ 数值特征已完成标准化。
预处理后，特征资料的形状: (210000, 28)


In [11]:
print("\n🤖 开始训练模型...")
X_train, X_test, y_train, y_test = train_test_split(
    X.values, y_encoded, test_size=0.3, random_state=42, stratify=y_encoded
)
# le 是我們之前用來編碼標籤的 LabelEncoder 物件
print("\n--- 正在驗證切分後的標籤分佈 ---")

unique_labels_train, counts_train = np.unique(y_train, return_counts=True)
print("\n訓練集 (y_train) 的標籤分佈:")
for label_int, count in zip(unique_labels_train, counts_train):
    original_label = le.inverse_transform([label_int])[0]
    print(f"  - {original_label}: {count} 筆")

unique_labels_test, counts_test = np.unique(y_test, return_counts=True)
print("\n測試集 (y_test) 的標籤分佈:")
for label_int, count in zip(unique_labels_test, counts_test):
    original_label = le.inverse_transform([label_int])[0]
    print(f"  - {original_label}: {count} 筆")

print("--------------------------------------------------\n")
# ==========================================================
print(f"训练集大小: {X_train.shape}")
print(f"测试集大小: {X_test.shape}")

results = {}

# --- 1. 训练 XGBoost (使用 GPU) ---
if device.type == 'cuda':
    print("\n--- 正在训练 XGBoost (GPU) ---")
    start_time = time.time()
    xgb_model = xgb.XGBClassifier(
        n_estimators=200,
        max_depth=8,
        learning_rate=0.1,
        tree_method='hist',  # 'hist' 在新版本中通常是 GPU 的好选择
        device='cuda',
        random_state=42
    )
    xgb_model.fit(X_train, y_train)
    xgb_time = time.time() - start_time

    y_pred_xgb = xgb_model.predict(X_test)
    xgb_accuracy = accuracy_score(y_test, y_pred_xgb)
    results['xgboost_gpu'] = {'time': xgb_time, 'accuracy': xgb_accuracy, 'model': xgb_model}
    print(f"✓ XGBoost 训练完成，耗时: {xgb_time:.2f} 秒，准确率: {xgb_accuracy:.4f}")
else:
    print("\n--- 跳过 XGBoost GPU 训练，因为未找到 GPU ---")


# --- 2. 训练 PyTorch 神经网络 (使用 GPU) ---
if device.type == 'cuda':
    print("\n--- 正在训练 PyTorch 神经网络 (GPU) ---")
    # 定义神经网络模型 (取自您的原始码)
    class FullNN(nn.Module):
        def __init__(self, input_size, hidden_size, output_size):
            super(FullNN, self).__init__()
            self.fc1 = nn.Linear(input_size, hidden_size)
            self.bn1 = nn.BatchNorm1d(hidden_size)
            self.relu = nn.ReLU()
            self.dropout = nn.Dropout(0.3)
            self.fc2 = nn.Linear(hidden_size, hidden_size // 2)
            self.bn2 = nn.BatchNorm1d(hidden_size // 2)
            self.fc3 = nn.Linear(hidden_size // 2, hidden_size // 4)
            self.bn3 = nn.BatchNorm1d(hidden_size // 4)
            self.fc4 = nn.Linear(hidden_size // 4, output_size)

        def forward(self, x):
            x = self.relu(self.bn1(self.fc1(x)))
            x = self.dropout(x)
            x = self.relu(self.bn2(self.fc2(x)))
            x = self.dropout(x)
            x = self.relu(self.bn3(self.fc3(x)))
            x = self.dropout(x)
            x = self.fc4(x)
            return x

    # 准备资料
    X_train_tensor = torch.FloatTensor(X_train.astype(np.float32)).to(device)
    y_train_tensor = torch.LongTensor(y_train).to(device)
    X_test_tensor = torch.FloatTensor(X_test.astype(np.float32)).to(device)
    y_test_tensor = torch.LongTensor(y_test).to(device)

    # 建立模型
    input_size = X_train.shape[1]
    output_size = len(le.classes_)
    nn_model = FullNN(input_size, 256, output_size).to(device)

    optimizer = optim.Adam(nn_model.parameters(), lr=0.001)
    criterion = nn.CrossEntropyLoss()

    # 训练模型
    start_time = time.time()
    nn_model.train()
    epochs = 50 # 您可以根据需要调整
    for epoch in range(epochs):
        optimizer.zero_grad()
        outputs = nn_model(X_train_tensor)
        loss = criterion(outputs, y_train_tensor)
        loss.backward()
        optimizer.step()
        if (epoch + 1) % 10 == 0:
            print(f"  Epoch {epoch+1}/{epochs}, Loss: {loss.item():.4f}")

    torch.cuda.synchronize() # 确保所有 GPU 操作完成
    nn_time = time.time() - start_time

    # 评估模型
    nn_model.eval()
    with torch.no_grad():
        outputs = nn_model(X_test_tensor)
        _, predicted = torch.max(outputs, 1)
        nn_accuracy = (predicted == y_test_tensor).float().mean().item()
    results['pytorch_nn_gpu'] = {'time': nn_time, 'accuracy': nn_accuracy, 'model': nn_model}
    print(f"✓ PyTorch NN 训练完成，耗时: {nn_time:.2f} 秒，准确率: {nn_accuracy:.4f}")
else:
    print("\n--- 跳过 PyTorch GPU 训练，因为未找到 GPU ---")


🤖 开始训练模型...

--- 正在驗證切分後的標籤分佈 ---

訓練集 (y_train) 的標籤分佈:
  - BP: 10500 筆
  - DoS: 10500 筆
  - FoT: 10500 筆
  - MitM: 10500 筆
  - Normal: 105000 筆

測試集 (y_test) 的標籤分佈:
  - BP: 4500 筆
  - DoS: 4500 筆
  - FoT: 4500 筆
  - MitM: 4500 筆
  - Normal: 45000 筆
--------------------------------------------------

训练集大小: (147000, 28)
测试集大小: (63000, 28)

--- 正在训练 XGBoost (GPU) ---
✓ XGBoost 训练完成，耗时: 2.68 秒，准确率: 0.9810

--- 正在训练 PyTorch 神经网络 (GPU) ---
  Epoch 10/50, Loss: 0.9722
  Epoch 20/50, Loss: 0.7242
  Epoch 30/50, Loss: 0.5793
  Epoch 40/50, Loss: 0.4824
  Epoch 50/50, Loss: 0.4127
✓ PyTorch NN 训练完成，耗时: 1.24 秒，准确率: 0.8960


In [12]:
print("\n💾 正在储存结果与模型...")
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# 储存预处理器
joblib.dump(scaler, os.path.join(output_folder_path, f'scaler_{timestamp}.pkl'))
joblib.dump(le, os.path.join(output_folder_path, f'label_encoder_{timestamp}.pkl'))
print(f"✓ 预处理器已储存至 {output_folder_path}")

# 储存模型
for name, result in results.items():
    if name == 'pytorch_nn_gpu':
        model_path = os.path.join(output_folder_path, f'{name}_{timestamp}.pth')
        torch.save(result['model'].state_dict(), model_path)
    else:
        model_path = os.path.join(output_folder_path, f'{name}_{timestamp}.pkl')
        joblib.dump(result['model'], model_path)
    print(f"✓ 模型 {name} 已储存至: {model_path}")

# 储存报告
report = {
    'training_date': datetime.now().isoformat(),
    'models_trained': list(results.keys()),
    'results': {
        name: {'time_seconds': res['time'], 'accuracy': res['accuracy']} for name, res in results.items()
    }
}
report_path = os.path.join(output_folder_path, f'training_report_{timestamp}.json')
with open(report_path, 'w') as f:
    json.dump(report, f, indent=2)
print(f"✓ 训练报告已储存至: {report_path}")

print("\n🎉 全部流程执行完毕！")



💾 正在储存结果与模型...
✓ 预处理器已储存至 /content/drive/MyDrive/BTAT/gpu_models_output/
✓ 模型 xgboost_gpu 已储存至: /content/drive/MyDrive/BTAT/gpu_models_output/xgboost_gpu_20250702_073122.pkl
✓ 模型 pytorch_nn_gpu 已储存至: /content/drive/MyDrive/BTAT/gpu_models_output/pytorch_nn_gpu_20250702_073122.pth
✓ 训练报告已储存至: /content/drive/MyDrive/BTAT/gpu_models_output/training_report_20250702_073122.json

🎉 全部流程执行完毕！
